In [1]:
import backtrader as bt
import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from hmmlearn.hmm import GaussianHMM
from hurst import compute_Hc
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [ ]:
def _fit_hmm_model(X, n_states):
    """Fits a Gaussian HMM and returns the model and log-likelihood."""
    model = GaussianHMM(
        n_components=n_states, 
        covariance_type="diag", 
        n_iter=1000, 
        random_state=42, 
        tol=1e-4, 
        min_covar=1e-6
    )
    model.fit(X)

    # 1. Log-Likelihood from EM Algorithm
    log_likelihood = model.score(X)
    return model, log_likelihood

def fit_hmm_regimes(returns, n_states):
    """
    Uses EM algorithm to fit HMMs from 2 to max_states.
    Selects the optimal number of states using BIC.
    """
    X = (returns.values * 100).reshape(-1, 1)
    n_samples = len(X)

    bic_records = []
    
    try:
        model, log_likelihood = _fit_hmm_model(X, n_states)
        
        # 2. Calculate Number of Free Parameters (k)
        # Transitions: n_states * (n_states - 1)
        # Means: n_states (since 1D data)
        # Variances: n_states
        # Initial Probabilities: n_states - 1
        k = (n_states**2 - n_states) + n_states + n_states + (n_states - 1)
        
        # 3. Calculate BIC: k * ln(n) - 2 * ln(L)
        bic = k * np.log(n_samples) - 2 * log_likelihood

        bic_records.append({
            'n_states': n_states,
            'log_likelihood': log_likelihood,
            'bic': bic
        })
            
    except Exception:
        # If EM fails to converge for a specific n_states, skip it
        pass
    
    bic_df = pd.DataFrame(bic_records)

    if bic_df.empty:
        return None, None

    return bic_df, model


def get_ordered_states(model, returns):
    """Maps HMM states to a consistent order based on volatility."""
    X = (returns.values * 100).reshape(-1, 1)
    hidden_states = model.predict(X)

    n_states = model.n_components
    
    # --- STANDARDIZATION ---
    # Extract variances and sort them from smallest to largest
    variances = np.array([np.diag(model.covars_[i])[0] for i in range(n_states)])
    sorted_idx = np.argsort(variances)
    
    # Create a mapping dictionary: old_state_id -> new_sorted_state_id
    state_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted_idx)}
    
    # Map the hidden states so 0 is ALWAYS lowest vol, and (N-1) is ALWAYS highest vol
    ordered_states = pd.Series(hidden_states).map(state_map).values
    
    return ordered_states

def calculate_rolling_hurst(log_return_series, window=100):
    """Calculates rolling Hurst Exponent using R/S."""
    hurst_values = np.full(len(log_return_series), np.nan)
    
    # Can be computationally heavy; loop is necessary for rolling DFA
    for i in range(window, len(log_return_series)):
        window_data = log_return_series.iloc[i-window:i].values
        try:
            # https://github.com/Mottl/hurst/blob/master/hurst/__init__.py
            H, _, _ = compute_Hc(window_data, kind='random_walk', simplified=True)
            hurst_values[i] = H
        except:
            hurst_values[i] = np.nan
            
    return hurst_values

def _apply_strategy_rules(df, model, n_states, hurst_window=100, momentum_periods=5):
    df['Regime'] = get_ordered_states(model, df['Log_Return'])
    
    # 2. Micro: Hurst Exponent
    df['Hurst'] = calculate_rolling_hurst(df['Log_Return'], window=hurst_window)
    
    # 3. Tactical: Momentum (5-days return)
    df['Momentum'] = df['Close'].pct_change(periods=momentum_periods)
    
    # 4. Signal Integration Logic (VECTORIZED for speed)
    df['Signal'] = 0 

    # Identify the absolute Lowest and Highest volatility regimes
    is_lowest_vol = df['Regime'] == 0
    is_highest_vol = df['Regime'] == (n_states - 1)
    
    high_hurst, low_hurst = df['Hurst'] > 0.55, df['Hurst'] < 0.45
    pos_mom, neg_mom = df['Momentum'] > 0, df['Momentum'] < 0

    # RULES (Middle states, if any, safely default to 0 signal)
    df.loc[is_lowest_vol & high_hurst & pos_mom, 'Signal'] = 1
    df.loc[is_lowest_vol & low_hurst, 'Signal'] = -np.sign(df['Momentum'])
    df.loc[is_highest_vol & high_hurst & neg_mom, 'Signal'] = -1

    # Clean up: Ensure NAs in Hurst/Momentum don't trigger signals
    df.loc[df['Hurst'].isna() | df['Momentum'].isna(), 'Signal'] = 0

    # Shift signal to avoid look-ahead bias
    df['Position'] = df['Signal'].shift(1).fillna(0)
    df['Strategy_Return'] = df['Position'] * df['Log_Return']

    return df

def process_single_asset(price_series, log_return_series, n_states):
    """Applies the strategy logic to a single asset."""
    # Drop leading/trailing NaNs to isolate the active trading period
    valid_price = price_series.dropna()
    valid_log_return = log_return_series.dropna()
    
    # Require minimum data points (e.g., 100 days) to run HMM + Rolling Hurst
    if len(valid_log_return) < 100 or len(valid_price) < 100:
        return pd.DataFrame(), pd.DataFrame() 

    df = pd.DataFrame({'Log_Return': valid_log_return, 'Close': valid_price})
    # fill the first NaN with 0 so HMM doesn't crash
    df.fillna(0, inplace=True)

    # 1. Macro: HMM Regime
    # Extract optimal states
    bic_df, model = fit_hmm_regimes(df, n_states=n_states)

    if model is None:
        return pd.DataFrame(), pd.DataFrame() # No valid models found, return empty DataFrame

    df = _apply_strategy_rules(df, model, n_states)
    
    # Return Hurst as well so we don't have to recalculate it later!
    return df[['Log_Return', 'Strategy_Return', 'Signal']], bic_df

def process_universe(df_prices, n_states):
    """Processes all assets and returns combined matrices."""
    print(f"Processing assets and extracting indicators for {n_states} states...")

    # Get sector and industry information for each ticker
    # This is necessary for the sector-neutral transformation later on
    tickers = pd.read_csv("/Users/jacsonchong/Documents/NUS/AY2025 2026 Semester 2/FE5214/Project/codebase/data_cn/tickers.csv", header=None)
    tickers.columns = ['ticker', 'gics']
    tickers['sector'] = tickers['gics'].apply(lambda x: str(x)[:4] if x is not None else None)
    tickers['industry'] = tickers['gics'].apply(lambda x: str(x)[:6] if x is not None else None)

    gics_map = tickers.set_index('ticker')['sector']

    log_return = np.log(df_prices / df_prices.shift(1))

    log_return_T = log_return.T

    # Add industry label
    log_return_T['sector'] = gics_map

    # Group by industry and subtract mean
    log_return_neutral_T = (
        log_return_T
        .groupby('sector')
        .transform(lambda x: x - x.mean())
    )

    df_log_return_neutral = log_return_neutral_T.T

    all_strat_ret, all_bh_ret, all_signals = {}, {}, {}
    all_bic = []
    
    for ticker in tqdm(df_prices.columns, desc="Processing Universe"):
        asset_data, bic_df = process_single_asset(df_prices[ticker], df_log_return_neutral[ticker], n_states)
        if not asset_data.empty:
            all_strat_ret[ticker] = asset_data['Strategy_Return']
            all_bh_ret[ticker] = asset_data['Log_Return']
            all_signals[ticker] = asset_data['Signal']

            bic_df['ticker'] = ticker
            all_bic.append(bic_df)
        else:
            print(f"Skipping {ticker} due to insufficient data.")
            
    return pd.DataFrame(all_strat_ret), pd.DataFrame(all_bh_ret), pd.DataFrame(all_signals), pd.concat(all_bic, ignore_index=True)

In [3]:
def calculate_mcap_weighted_returns(strat_log_returns, bh_log_returns, mcap_df):
    """Aggregates individual asset returns into a Market Cap weighted portfolio."""
    print("Calculating Market Cap weighted portfolio returns...")
    
    mcap_df = mcap_df.reindex(index=bh_log_returns.index, columns=bh_log_returns.columns)
    shifted_mcap = mcap_df.shift(1) # Prevent look-ahead bias
    
    valid_returns_mask = ~bh_log_returns.isna()
    active_mcap = shifted_mcap.where(valid_returns_mask)
    
    weights = active_mcap.div(active_mcap.sum(axis=1), axis=0).fillna(0)
    
    simple_bh = np.exp(bh_log_returns) - 1
    simple_strat = np.exp(strat_log_returns) - 1
    
    port_simple_bh = (simple_bh * weights).sum(axis=1)
    port_simple_strat = (simple_strat * weights).sum(axis=1)
    
    port_log_bh = np.log(1 + port_simple_bh)
    port_log_strat = np.log(1 + port_simple_strat)
    
    return pd.DataFrame({'Log_Return': port_log_bh, 'Strategy_Return': port_log_strat}).replace(0, np.nan).dropna()


In [4]:
def evaluate_backtest(df, risk_free_rate=0.02):
    """Calculates standard quantitative performance metrics."""
    df['Simple_Market_Ret'] = np.exp(df['Log_Return']) - 1
    df['Simple_Strat_Ret'] = np.exp(df['Strategy_Return']) - 1
    
    df['Cum_Market'] = (1 + df['Simple_Market_Ret']).cumprod()
    df['Cum_Strat'] = (1 + df['Simple_Strat_Ret']).cumprod()
    
    trading_days = len(df.dropna())
    years = trading_days / 252
    
    cagr_market = (df['Cum_Market'].iloc[-1]) ** (1 / years) - 1
    cagr_strat = (df['Cum_Strat'].iloc[-1]) ** (1 / years) - 1
    
    vol_market = df['Simple_Market_Ret'].std() * np.sqrt(252)
    vol_strat = df['Simple_Strat_Ret'].std() * np.sqrt(252)
    
    sharpe_market = (cagr_market - risk_free_rate) / vol_market
    sharpe_strat = (cagr_strat - risk_free_rate) / vol_strat
    
    mdd_market = ((df['Cum_Market'] - df['Cum_Market'].cummax()) / df['Cum_Market'].cummax()).min()
    mdd_strat = ((df['Cum_Strat'] - df['Cum_Strat'].cummax()) / df['Cum_Strat'].cummax()).min()

    results = pd.DataFrame({
        'Metric': ['CAGR', 'Ann. Volatility', 'Sharpe Ratio', 'Max Drawdown'],
        'Benchmark (Buy & Hold)': [f"{cagr_market*100:.2f}%", f"{vol_market*100:.2f}%", f"{sharpe_market:.2f}", f"{mdd_market*100:.2f}%"],
        'HMM-Hurst Strategy': [f"{cagr_strat*100:.2f}%", f"{vol_strat*100:.2f}%", f"{sharpe_strat:.2f}", f"{mdd_strat*100:.2f}%"]
    }).set_index('Metric')
    
    return results


In [5]:
# 1. Load Price Data and Market Cap Data
csv_file_path = "/Users/jacsonchong/Documents/NUS/AY2025 2026 Semester 2/FE5214/Project/codebase/data_cn/adjusted.csv"
df_prices = pd.read_csv(csv_file_path)
df_prices['Date'] = pd.to_datetime(df_prices['Date'], format='%Y%m%d')
df_prices.set_index('Date', inplace=True)

mcap_file_path = "/Users/jacsonchong/Documents/NUS/AY2025 2026 Semester 2/FE5214/Project/codebase/data_cn/mktcap.csv" 
df_mcap = pd.read_csv(mcap_file_path)
df_mcap['Date'] = pd.to_datetime(df_mcap['Date'], format='%Y%m%d')
df_mcap.set_index('Date', inplace=True)


In [6]:
# 2. Run the Main Engine (Extract Returns and Hurst)
universe_strat_returns_2, universe_bh_returns_2, universe_signal_df_2, bic_all_2 = process_universe(df_prices, n_states=2)
universe_strat_returns_3, universe_bh_returns_3, universe_signal_df_3, bic_all_3 = process_universe(df_prices, n_states=3)

Processing assets and extracting indicators...


Processing Universe:   4%|▍         | 35/915 [00:45<19:01,  1.30s/it]

Skipping 000406 due to insufficient data.


Processing Universe:   8%|▊         | 77/915 [01:38<18:48,  1.35s/it]

Skipping 000618 due to insufficient data.


...

Processing Universe:  98%|█████████▊| 900/915 [14:38<00:06,  2.41it/s]

Skipping 688126 due to insufficient data.


Processing Universe:  99%|█████████▊| 902/915 [14:38<00:04,  2.87it/s]

Skipping 688187 due to insufficient data.
Skipping 688223 due to insufficient data.


Processing Universe: 100%|██████████| 915/915 [14:42<00:00,  1.04it/s]


In [7]:
print("Running BIC analysis...")
bic_all = pd.concat([bic_all_2, bic_all_3], ignore_index=True)
bic_summary = bic_all.groupby('n_states')['bic'].agg(['mean', 'std', 'count'])

best_model = bic_all.loc[bic_all.groupby('ticker')['bic'].idxmin()]
selection_freq = best_model['n_states'].value_counts(normalize=True)

print("\nBIC Summary:")
print(bic_summary)

print("\nModel Selection Frequency:")
print(selection_freq)

Running BIC analysis...

BIC Summary:
                  mean           std  count
n_states                                   
2         74076.867490  23065.923459    880
3         72849.531809  22607.881787    662

Model Selection Frequency:
n_states
3    0.678532
2    0.321468
Name: proportion, dtype: float64


In [8]:
# 3. Create Aggregated Portfolio (Market Cap Weighted)
# Average the cross-sectional log returns to simulate a daily marketcap-weight portfolio
portfolio_eval_df = calculate_mcap_weighted_returns(
    strat_log_returns=universe_strat_returns_3,
    bh_log_returns=universe_bh_returns_3,
    mcap_df=df_mcap
)
# Run the backtest evaluator
performance_metrics = evaluate_backtest(portfolio_eval_df)

print("\n--- Strategy vs Benchmark Performance ---")
print(performance_metrics)
print("-" * 40)

Calculating Market Cap weighted portfolio returns...

--- Strategy vs Benchmark Performance ---
                Benchmark (Buy & Hold) HMM-Hurst Strategy
Metric                                                   
CAGR                             0.11%              5.26%
Ann. Volatility                  8.62%              3.29%
Sharpe Ratio                     -0.22               0.99
Max Drawdown                   -44.47%             -5.21%
----------------------------------------


# Use Backtrader

In [9]:
def generate_target_weights(signals_df, df_mcap, is_buy_and_hold=False):
    """
    Converts signals and market caps into target portfolio weights.
    If is_buy_and_hold=True, it ignores signals and just buys the market-cap weighted index.
    """
    print("Generating Target Weights for Backtrader...")

    # Align Market Cap with Signals
    df_mcap = df_mcap.reindex(index=signals_df.index, columns=signals_df.columns)
    execution_mcap = df_mcap.shift(1) # Base weight on yesterday's Mcap
    
    if is_buy_and_hold:
        # B&H is always invested (Signal = 1) for all active assets
        execution_signals = signals_df.copy()
        execution_signals[:] = 1 
    else:
        # Strategy shifts signals by 1 day to prevent look-ahead bias
        execution_signals = signals_df.shift(1).fillna(0)
    
    # Only calculate weights for assets we actually want to trade today
    active_mcap = execution_mcap.where(execution_signals != 0)
    
    # Normalize the weights so they sum to 1.0 (100% of portfolio)
    normalized_weights = active_mcap.div(active_mcap.sum(axis=1), axis=0).fillna(0)
    
    # Multiply by the signal to get directional weights (Strategy uses -1/1, B&H uses 1)
    target_weights = normalized_weights * execution_signals
    
    return target_weights

In [ ]:
# =========================================================
# 2. BACKTRADER CUSTOM DATA FEED & STRATEGY
# =========================================================

class SignalData(bt.feeds.PandasData):
    """Custom Data Feed that includes our pre-calculated Target Weight."""
    lines = ('target_weight',)
    params = (
        ('datetime', None),
        ('open', 'Close'),
        ('high', 'Close'),
        ('low', 'Close'),
        ('close', 'Close'),
        ('volume', -1),
        ('openinterest', -1),
        ('target_weight', 'Target_Weight'), 
    )

class DetailedExecutionStrategy(bt.Strategy):
    """Executes trades and tracks deeply detailed logging."""
    
    params = (
        ('logger', None),
        ('print_logs', False), # Set to True to see daily logs (WARNING: Massive output)
    )

    def __init__(self):
        self.logger = self.p.logger
    
    def log(self, txt, dt=None):
        """Logging function for trade tracking"""
        if self.p.print_logs:
            dt = dt or self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()} | {txt}')

    def notify_order(self, order):
        """Triggered when an order is executed (Buy/Sell/Size)."""
        if order.status in [order.Submitted, order.Accepted]:
            return # Awaiting execution

        if order.status in [order.Completed]:                
            if order.isbuy():
                msg = (f'Datetime: {self.datas[0].datetime.date(0)} | BUY EXECUTED  | Ticker: {order.data._name} | Price: ${order.executed.price:.2f} | Size: {order.executed.size}')
            elif order.issell():
                msg = (f'Datetime: {self.datas[0].datetime.date(0)} | SELL EXECUTED | Ticker: {order.data._name} | Price: ${order.executed.price:.2f} | Size: {order.executed.size}')

            if self.logger:
                self.logger.info(msg)
                
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f'ORDER FAILED  | Ticker: {order.data._name} | Status: {order.getstatusname()}')

    def notify_trade(self, trade):
        """Triggered when a trade is closed (calculates PnL)."""
        if not trade.isclosed:
            return
        msg = (f'Datetime: {self.datas[0].datetime.date(0)} | TRADE CLOSED. | Ticker: {trade.data._name} | Gross PnL: ${trade.pnl:.2f} | Net PnL: ${trade.pnlcomm:.2f}')
        if self.logger:
            self.logger.info(msg)

    def next(self):
        """Executes the portfolio rebalancing daily."""
        for data in self.datas:
            target_pct = data.target_weight[0]
            
            # Backtrader natively handles the math: 
            # (Current Cash + Portfolio Value) * target_pct = Target Asset Value
            if not np.isnan(target_pct):
                self.order_target_percent(data, target=target_pct)


In [ ]:
# =========================================================
# 3. BACKTRADER EXECUTION ENGINE
# =========================================================

def run_backtrader_engine(df_prices, target_weights_df, test_name="Strategy", starting_cash=1_000_000.0, logger=None,print_logs=False):
    """Initializes Cerebro, runs the engine, and calculates KPIs."""
    print(f"\nInitializing Backtrader Engine for: {test_name} (This may take a minute for 900+ assets) ...")
    
    cerebro = bt.Cerebro()
    cerebro.broker.setcash(starting_cash)
    
    # Add a realistic commission (e.g., 0.1%)
    cerebro.broker.setcommission(commission=0.001)

    # Add Data Feeds for all 916 assets
    for ticker in df_prices.columns:
        # Create a single DataFrame for this specific asset
        asset_df = pd.DataFrame({
            'Close': df_prices[ticker],
            'Target_Weight': target_weights_df[ticker]
        }).dropna(subset=['Close']) # Drop entirely empty dates for this asset
        
        # Skip if the asset has no data
        if asset_df.empty: continue
            
        # Convert to Backtrader Custom Data Feed
        data = SignalData(dataname=asset_df, name=ticker)
        cerebro.adddata(data)

    # cerebro.addstrategy(HMMHurstExecution)
    cerebro.addstrategy(DetailedExecutionStrategy, logger=logger, print_logs=False) # Set print_logs=True for verbose output
    
    # Add Analyzers to track metrics
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name="trades")
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name="drawdown")
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name="sharpe", riskfreerate=0.02, annualize=True)
    cerebro.addanalyzer(bt.analyzers.TimeReturn, _name="returns")
    cerebro.addanalyzer(bt.analyzers.Transactions, _name="transactions")

    logger.info("Starting Backtest execution...")
    logger.info(f'Starting Portfolio Value: ${cerebro.broker.getvalue():.2f}')
    
    results = cerebro.run()
    strat = results[0]
    
    # Extract KPIs
    final_value = cerebro.broker.getvalue()
    net_pnl = final_value - starting_cash

    # print(f'Final Portfolio Value:  ${cerebro.broker.getvalue():.2f}')
    
    # --- PRINT TRADE LEDGER ---
    trade_analysis = strat.analyzers.trades.get_analysis()
    total_closed = trade_analysis.total.closed if 'total' in trade_analysis else 0
    won_trades = trade_analysis.won.total if 'won' in trade_analysis else 0
    win_rate = (won_trades / total_closed * 100) if total_closed > 0 else 0

    drawdown_analysis = strat.analyzers.drawdown.get_analysis()
    max_dd = drawdown_analysis.max.drawdown
    
    sharpe_analysis = strat.analyzers.sharpe.get_analysis()
    sharpe_ratio = sharpe_analysis['sharperatio'] if sharpe_analysis['sharperatio'] is not None else 0

    txn = strat.analyzers.transactions.get_analysis()

    # Convert dict → DataFrame
    txn_df = pd.DataFrame([
        {
            'datetime': k,
            'ticker': v[0][0],
            'size': v[0][1],
            'price': v[0][2]
        }
        for k, v in txn.items()
    ])

    txn_df.to_csv(f"{test_name}_transactions.csv", index=False)

    # Print Formatted Results
    logger.info("\n" + "="*50)
    logger.info(f"BACKTRADER RESULTS: {test_name.upper()}")
    logger.info("="*50)
    logger.info(f"Final Portfolio Value : ${final_value:,.2f}")
    logger.info(f"Net PnL               : ${net_pnl:,.2f}")
    logger.info(f"Total Trades Closed   : {total_closed:,}")
    logger.info(f"Win Rate (Strike Rate): {win_rate:.2f}%")
    logger.info(f"Max Drawdown          : {max_dd:.2f}%")
    logger.info(f"Sharpe Ratio          : {sharpe_ratio:.2f}")
    logger.info("="*50)

    return cerebro, strat

In [13]:
def setup_logger(test_name):
    logger = logging.getLogger(test_name)
    logger.setLevel(logging.INFO)

    # Avoid duplicate handlers if rerun
    if logger.hasHandlers():
        logger.handlers.clear()

    # File handler
    fh = logging.FileHandler(f"{test_name}.log")
    fh.setLevel(logging.INFO)

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    # Format
    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s'
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    return logger

In [14]:
# 2. Calculate the Backtrader Target Weights
target_weights = generate_target_weights(universe_signal_df_3, df_mcap)

Generating Target Weights for Backtrader...


In [ ]:
# 3. Pass it to the Backtrader Engine!
common_columns = list(set(df_prices.columns).intersection(set(target_weights.columns)))

# ---------------------------------------------------------
# TEST 1: BUY & HOLD BENCHMARK
# ---------------------------------------------------------
# Generate B&H weights (ignores signals, just holds market cap weights)
test_name = "Buy & Hold (Market Cap Weighted)"
logger = setup_logger(test_name)

bh_weights = generate_target_weights(
    signals_df=universe_signal_df_3, 
    df_mcap=df_mcap, 
    is_buy_and_hold=True
)

run_backtrader_engine(
    df_prices=df_prices[common_columns], 
    target_weights_df=bh_weights[common_columns], 
    test_name=test_name, 
    logger=logger,
    print_logs=False 
)

Generating Target Weights for Backtrader...

Initializing Backtrader Engine for: Buy & Hold (Market Cap Weighted) (This may take a minute for 900+ assets) ...
Starting Backtest execution...
Starting Portfolio Value: $1000000.00


In [ ]:
# ---------------------------------------------------------
# TEST 2: HMM-HURST STRATEGY
# ---------------------------------------------------------
# Generate Strategy weights (applies your -1/0/1 signals)
test_name = "HMM & Hurst Strategy"
logger = setup_logger(test_name)

strat_weights = generate_target_weights(
    signals_df=universe_signal_df_3, 
    df_mcap=df_mcap, 
    is_buy_and_hold=False
)

run_backtrader_engine(
    df_prices=df_prices[common_columns], 
    target_weights_df=strat_weights[common_columns], 
    test_name="HMM & Hurst Strategy", 
    logger=logger,
    print_logs=False # Set to True if you want to see the microscopic trade details!
)

# Running Full Study (to verify if lower BIC improve metrics)

In [ ]:
# from scipy.stats import ttest_rel

# def fit_hmm_regimes_with_states(returns, num_states):
#     X = (returns.values * 100).reshape(-1, 1)

#     try:
#         model, _ = _fit_hmm_model(X, num_states)

#         return model
#     except Exception:
#         pass

#     return None

# def process_asset_with_states(price_series, n_states):
#     valid_prices = price_series.dropna()

#     if len(valid_prices) < 100:
#         return None

#     df = pd.DataFrame({'Close': valid_prices})

#     df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1)).fillna(0)

#     # --- HMM ---
#     model = fit_hmm_regimes_with_states(df['Log_Return'], num_states=n_states)

#     if model is None:
#         return None
    
#     df = _apply_strategy_rules(df, model, n_states)

#     return df[['Log_Return', 'Strategy_Return']]

# def run_universe_backtest(df_prices, n_states):
#     strat_returns = []
#     bh_returns = []

#     for ticker in df_prices.columns:
#         res = process_asset_with_states(df_prices[ticker], n_states)

#         if res is None:
#             continue

#         strat_returns.append(res['Strategy_Return'])
#         bh_returns.append(res['Log_Return'])

#     strat_df = pd.concat(strat_returns, axis=1)
#     bh_df = pd.concat(bh_returns, axis=1)

#     return strat_df, bh_df

# def compare_strategies(df_prices, df_mcap):
#     # Run strategies
#     strat2, bh2 = run_universe_backtest(df_prices, 2)
#     portfolio_eval_df = calculate_mcap_weighted_returns(
#         strat_log_returns=strat2,
#         bh_log_returns=bh2,
#         mcap_df=df_mcap
#     )
#     # Run the backtest evaluator
#     performance_metrics = evaluate_backtest(portfolio_eval_df)
#     print("\n--- Strategy vs Benchmark Performance for 2 Regimes ---")
#     print(performance_metrics)
#     print("-" * 40)

#     strat3, bh3 = run_universe_backtest(df_prices, 3)
#     portfolio_eval_df = calculate_mcap_weighted_returns(
#         strat_log_returns=strat3,
#         bh_log_returns=bh3,
#         mcap_df=df_mcap
#     )
#     # Run the backtest evaluator
#     performance_metrics = evaluate_backtest(portfolio_eval_df)
#     print("\n--- Strategy vs Benchmark Performance for 3 Regimes ---")
#     print(performance_metrics)
#     print("-" * 40)

#     # Compute per-asset Sharpe
#     def sharpe(x, risk_free_rate = 0.02):
#         return (x.mean() - risk_free_rate) / x.std() * np.sqrt(252) if x.std() > 0 else 0

#     sharpe_2 = strat2.apply(lambda x: sharpe(x, risk_free_rate=0.02))
#     sharpe_3 = strat3.apply(lambda x: sharpe(x, risk_free_rate=0.02))

#     # Paired t-test
#     t_stat, p_val = ttest_rel(sharpe_3, sharpe_2)

#     return sharpe_2, sharpe_3, t_stat, p_val

# def run_full_study(df_prices, df_mcap):
#     print("\nRunning strategy comparison if we allow 2 or 3 regimes...")
#     sharpe_2, sharpe_3, t_stat, p_val = compare_strategies(df_prices, df_mcap)

#     print("\nStatistical Test (3-state vs 2-state):")
#     print(f"T-stat: {t_stat:.4f}, P-value: {p_val:.4f}")

#     return {
#         'bic_summary': bic_summary,
#         'selection_freq': selection_freq,
#         'sharpe_2': sharpe_2,
#         'sharpe_3': sharpe_3,
#         't_stat': t_stat,
#         'p_val': p_val
#     }

In [ ]:
# run_full_study(df_prices, df_mcap)